# PPAD: Patch Predictive Anomaly Detection

This notebook provides a self-contained, interactive version of the **PPAD (Patch Predictive Anomaly Detection)** codebase.
It supports training and zero-shot anomaly detection on three benchmark datasets:
- **MVTec AD**
- **MVTec AD 2**
- **VisA**

### Core Concept
PPAD detects anomalies using a predictive patch coding framework: if an image is normal, any local patch should be predictable from its surrounding context. Specifically:
1. **hp** = Encoder(crop(image, patch_i)) &rarr; actual patch embedding
2. **hi** = Encoder(image with patch_i masked out) &rarr; context embedding
3. **ho** = Transformer_Predictor(hi, pos_i) &rarr; predicted patch embedding
4. **anomaly_score** = 1 - cosine_similarity(hp, ho)

Normal patches yield low scores because the predictor can easily guess them from the context. Anomalous patches break this predictability, yielding high scores.

### Running on Kaggle
1. **Add the Dataset(s)**: Click **+ Add Input** in the top-right of your Kaggle notebook, search for your dataset(s) (e.g., MVTec AD, VisA), and add them.
2. **Configure Paths**: In the **Configuration** cell below, set your training and evaluation dataset names and paths. For cross-dataset zero-shot experiments (e.g. train on MVTec, test on MVTec 2), you can configure them separately.
3. **Run All**: Execute the cells to train the predictor and run evaluation.

In [ ]:
# Install dependencies if they are not already installed
# !pip install -q torch torchvision scikit-learn tqdm pillow numpy matplotlib

import os
import json
from pathlib import Path
from typing import List, Union

from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, ConcatDataset, DataLoader
import torchvision.transforms as T
from tqdm.notebook import tqdm

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from sklearn.metrics import roc_auc_score, precision_recall_curve

print("Imports successful.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
class Config:
    # =========================================================================
    # 1. Training Dataset Configuration (Used when Config.train = True)
    # =========================================================================
    # Dataset: 'mvtec' | 'mvtec2' | 'visa'
    train_dataset = 'mvtec'
    
    # Path to training dataset root (adjust based on Kaggle mount point)
    train_data_path = '/kaggle/input/mvtec-ad'
    
    # Category: single category (e.g. 'bottle'), 'all', or comma-separated list (e.g. 'bottle,cable')
    train_category = 'all'
    
    # =========================================================================
    # 2. Evaluation Dataset Configuration (Used when Config.evaluate = True)
    # =========================================================================
    # Dataset: 'mvtec' | 'mvtec2' | 'visa'
    eval_dataset = 'mvtec2'
    
    # Path to evaluation dataset root (adjust based on Kaggle mount point)
    eval_data_path = '/kaggle/input/mvtec-ad-2'
    
    # Category: single category (e.g. 'sheet_metal'), 'all', or comma-separated list
    eval_category = 'all'
    
    # Optional override for loaded checkpoint tag. 
    # If None, it automatically maps to the checkpoint trained in this session:
    #   - Single category train: '<train_dataset>/<train_category>'
    #   - Multi-category train: '<train_dataset>/all'
    # Specify a custom string (e.g., 'mvtec/all') to load a pre-trained model.
    ckpt_tag = None
    
    # =========================================================================
    # 3. Hyperparameters & Architecture
    # =========================================================================
    patch_grid = 4            # 4x4 = 16 patches per image
    img_size = 224            # Input resolution
    encoder = 'dinov2_vits14' # DINOv2 model variant ('dinov2_vits14' or 'dinov2_vitb14')
    
    epochs = 50               # Number of training epochs
    batch_size = 8            # Batch size
    lr = 1e-4                 # Learning rate
    
    # =========================================================================
    # 4. Output Configuration
    # =========================================================================
    output_dir = './checkpoints'     # Directory to save model weights
    vis_dir = './visualizations'     # Directory to save test heatmaps
    
    # If False, will save full test heatmaps to `vis_dir`.
    # Regardless of this setting, the notebook will display the first 3 test samples inline.
    no_vis = False
    
    # Actions to perform
    train = True
    evaluate = True
    
    # Device to run on
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {Config.device}")
print(f"Train setup: dataset={Config.train_dataset} | category={Config.train_category} | path={Config.train_data_path}")
print(f"Eval setup: dataset={Config.eval_dataset} | category={Config.eval_category} | path={Config.eval_data_path}")

## 1. Model Definitions

Here we define the core network classes:
- `DINOv2Encoder`: Wraps the frozen DINOv2 backbone to extract CLS patch representations.
- `PatchPredictor`: A lightweight 2-layer Transformer encoder that predicts target patch embedding based on masked context and positional encoding.
- `PPAD`: The top-level class wrapping the pipeline, exposing methods to crop patches, mask contexts, perform forward passes, and upsample patch anomaly maps.

In [ ]:
"""
model.py - PPAD: Patch Predictive Anomaly Detection

Pipeline for each patch i in image x:
  hp = Encoder(crop(x, i))          # isolated patch embedding
  hi = Encoder(mask(x, i))          # context embedding (image with patch i zeroed)
  ho = Predictor(hi, pos_i)         # predicted patch embedding
  score_i = 1 - cosine_sim(hp, ho)  # anomaly score for patch i
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Frozen DINOv2 encoder
# ---------------------------------------------------------------------------

class DINOv2Encoder(nn.Module):
    """Wraps a frozen DINOv2 model; returns the CLS token."""

    def __init__(self, model_name: str = 'dinov2_vits14'):
        super().__init__()
        self.model = torch.hub.load('facebookresearch/dinov2', model_name)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.embed_dim: int = self.model.embed_dim

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, 3, 224, 224] normalized → returns [B, D]"""
        return self.model(x)


# ---------------------------------------------------------------------------
# Transformer predictor
# ---------------------------------------------------------------------------

class PatchPredictor(nn.Module):
    """
    Predicts the embedding of a target patch given:
      - hi  : context embedding [B, D]  (image with that patch masked)
      - pos : patch index       [B]     (integer, learned embedding table)

    The 2-token sequence [context | query] is fed through a small
    TransformerEncoder.  The output at the query position is projected
    to the final predicted embedding ho [B, D].
    """

    def __init__(self, embed_dim: int, num_patches: int,
                 num_heads: int = 6, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.pos_embed = nn.Embedding(num_patches, embed_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,          # pre-norm for training stability
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.out_proj = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, hi: torch.Tensor, patch_idx: torch.Tensor) -> torch.Tensor:
        pos = self.pos_embed(patch_idx)          # [B, D]
        seq = torch.stack([hi, pos], dim=1)      # [B, 2, D]
        out = self.transformer(seq)              # [B, 2, D]
        return self.out_proj(out[:, 1])          # [B, D]  ← query token output


# ---------------------------------------------------------------------------
# Full PPAD model
# ---------------------------------------------------------------------------

class PPAD(nn.Module):
    """
    Patch Predictive Anomaly Detection.

    Args:
        patch_grid  : patches per spatial dimension (patch_grid x patch_grid total)
        img_size    : input image size (square)
        encoder_name: DINOv2 variant (dinov2_vits14 / dinov2_vitb14 / ...)
    """

    def __init__(self, patch_grid: int = 4, img_size: int = 224,
                 encoder_name: str = 'dinov2_vits14'):
        super().__init__()
        self.patch_grid  = patch_grid
        self.img_size    = img_size
        self.num_patches = patch_grid * patch_grid
        self.patch_size  = img_size // patch_grid   # pixels per patch side

        self.encoder   = DINOv2Encoder(encoder_name)
        self.predictor = PatchPredictor(
            embed_dim   = self.encoder.embed_dim,
            num_patches = self.num_patches,
        )

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def _patch_coords(self, idx: int):
        """Return (r0, r1, c0, c1) pixel slice for patch index idx."""
        ps = self.patch_size
        r  = idx // self.patch_grid
        c  = idx  % self.patch_grid
        return r * ps, (r + 1) * ps, c * ps, (c + 1) * ps

    def _encode_all_patches(self, images: torch.Tensor) -> torch.Tensor:
        """
        Crop every patch from every image, resize to img_size, encode.
        images : [B, 3, H, W]
        returns: [B, N, D]
        """
        B, N = images.shape[0], self.num_patches
        crops = []
        for idx in range(N):
            r0, r1, c0, c1 = self._patch_coords(idx)
            p = images[:, :, r0:r1, c0:c1]                         # [B, 3, ps, ps]
            p = F.interpolate(p, self.img_size, mode='bilinear', align_corners=False)
            crops.append(p)                                          # [B, 3, H, W]

        # Stack → [B*N, 3, H, W] for one batched encoder call
        crops_cat = torch.cat(crops, dim=0)                         # [B*N, 3, H, W]
        hp_cat    = self.encoder(crops_cat)                         # [B*N, D]
        return hp_cat.view(N, B, -1).permute(1, 0, 2)              # [B, N, D]

    def _encode_all_contexts(self, images: torch.Tensor) -> torch.Tensor:
        """
        For each patch position, zero it out and encode the resulting image.
        images : [B, 3, H, W]
        returns: [B, N, D]
        """
        B, N = images.shape[0], self.num_patches
        masked_list = []
        for idx in range(N):
            r0, r1, c0, c1 = self._patch_coords(idx)
            m = images.clone()
            m[:, :, r0:r1, c0:c1] = 0.0   # 0 ≈ ImageNet mean after normalization
            masked_list.append(m)           # [B, 3, H, W]

        masked_cat = torch.cat(masked_list, dim=0)                  # [B*N, 3, H, W]
        hi_cat     = self.encoder(masked_cat)                       # [B*N, D]
        return hi_cat.view(N, B, -1).permute(1, 0, 2)              # [B, N, D]

    # ------------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------------

    def forward(self, images: torch.Tensor):
        """
        images: [B, 3, H, W] (normalized)
        Returns patch_scores [B, N] — higher = more anomalous.
        Only the predictor is differentiable; encoder is frozen.
        """
        B, N = images.shape[0], self.num_patches

        hp = self._encode_all_patches(images)    # [B, N, D]  — no grad
        hi = self._encode_all_contexts(images)   # [B, N, D]  — no grad

        # Flatten spatial dims for predictor
        hi_flat  = hi.reshape(B * N, -1)                                # [B*N, D]
        idx_flat = torch.arange(N, device=images.device).repeat(B)      # [B*N]
        ho_flat  = self.predictor(hi_flat, idx_flat)                    # [B*N, D]

        ho = ho_flat.view(B, N, -1)                                     # [B, N, D]

        # 1 - cosine similarity as anomaly score
        scores = 1.0 - F.cosine_similarity(hp, ho, dim=-1)             # [B, N]
        return scores

    def get_anomaly_map(self, images: torch.Tensor) -> torch.Tensor:
        """
        Upsample patch scores to a per-pixel heatmap.
        images: [B, 3, H, W]
        Returns: [B, 1, H, W]
        """
        scores = self.forward(images)                                   # [B, N]
        H, W   = images.shape[2], images.shape[3]
        grid   = scores.view(-1, 1, self.patch_grid, self.patch_grid)  # [B, 1, g, g]
        return F.interpolate(grid, size=(H, W), mode='bilinear', align_corners=False)


## 2. Dataset Loaders

Here we define custom `Dataset` classes for:
- **MVTec AD** (`MVTecDataset`)
- **MVTec AD 2** (`MVTec2Dataset`)
- **VisA** (`VisADataset`)
- Multi-category shared training (`MultiCategoryDataset`)

We also define `make_dataset`, which acts as a unified factory to load the datasets and split them into train (good images only) and test splits.

In [ ]:
"""
dataset.py - Dataset loaders for MVTec AD, MVTec AD 2, and VisA.

All datasets expose the same __getitem__ contract:
    image : [3, H, W]  normalized float tensor
    mask  : [1, H, W]  binary float tensor  (0 = normal, 1 = anomaly)
    label : int        0 = normal, 1 = anomaly

Use make_dataset() to get the right loader by name.
"""
from typing import List, Union
import os
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, ConcatDataset
import torchvision.transforms as T

MVTEC_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper',
]

MVTEC2_CATEGORIES = [
    'sheet_metal', 'vial', 'fabric', 'fruit_jelly',
    'rice_bag', 'wallplugs', 'walnuts', 'capacitor',
]

VISA_CATEGORIES = [
    'capsules', 'candle', 'pcb1', 'pcb2', 'pcb3', 'pcb4',
    'macaroni1', 'macaroni2', 'cashew', 'chewinggum', 'fryum', 'pipe_fryum',
]

IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]


def get_transform(img_size=224):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ])


class MVTecDataset(Dataset):
    """
    split='train' : only normal (good) images — used to train the predictor.
    split='test'  : all test images with labels and optional GT masks.
    """

    def __init__(self, root: str, category: str, split: str = 'train', img_size: int = 224):
        self.root = Path(root)
        self.transform = get_transform(img_size)
        self.mask_transform = T.Compose([
            T.Resize((img_size, img_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor(),
        ])
        self.samples = []   # (img_path, mask_path_or_None, label)
        self._load(split, category)

    def _load(self, split, category):
        split_dir = self.root / category / split
        if split == 'train':
            good_dir = split_dir / 'good'
            for p in sorted(good_dir.glob('*')):
                if p.suffix.lower() in ('.png', '.jpg', '.bmp'):
                    self.samples.append((p, None, 0))
        else:
            for defect_dir in sorted(split_dir.iterdir()):
                if not defect_dir.is_dir():
                    continue
                label = 0 if defect_dir.name == 'good' else 1
                for p in sorted(defect_dir.glob('*')):
                    if p.suffix.lower() not in ('.png', '.jpg', '.bmp'):
                        continue
                    mask = None
                    if label == 1:
                        mask = (self.root / category / 'ground_truth'
                                / defect_dir.name / (p.stem + '_mask.png'))
                        if not mask.exists():
                            mask = None
                    self.samples.append((p, mask, label))

        if len(self.samples) == 0:
            msg = f"No samples found for category '{category}', split '{split}' under split dir: {split_dir}\n"
            if not self.root.exists():
                msg += f"Root directory '{self.root}' does not exist. Please check your data_path setting."
            elif not (self.root / category).exists():
                subdirs = [p.name for p in self.root.iterdir() if p.is_dir() and not p.name.startswith('.')]
                msg += f"Category directory '{category}' not found under root '{self.root}'. Available directories: {subdirs}"
            elif not split_dir.exists():
                msg += f"Split directory '{split}' not found under '{self.root / category}'."
            else:
                if split == 'train':
                    good_dir = split_dir / 'good'
                    if not good_dir.exists():
                        msg += f"Good directory '{good_dir}' does not exist."
                    else:
                        all_files = [p.name for p in good_dir.glob('*') if not p.name.startswith('.')][:5]
                        msg += f"Good directory exists but no matching files found. Checked extensions: png, jpg, bmp. Found files: {all_files}"
                else:
                    msg += f"No test classes or files found under '{split_dir}'."
            raise ValueError(msg)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]
        image = self.transform(Image.open(img_path).convert('RGB'))
        H, W = image.shape[1], image.shape[2]
        if mask_path is not None:
            mask = self.mask_transform(Image.open(mask_path).convert('L'))
            mask = (mask > 0.5).float()
        else:
            mask = torch.zeros(1, H, W)
        return image, mask, label


class MultiCategoryDataset(Dataset):
    """
    Combines normal (good) training images from multiple MVTec categories
    into a single dataset so that one shared model can be trained on all.

    Only supports split='train' (normal images only).
    """

    def __init__(self, root: str, categories: List[str], img_size: int = 224):
        self.samples: list = []
        for cat in categories:
            cat_ds = MVTecDataset(root, cat, split='train', img_size=img_size)
            self.samples.extend(cat_ds.samples)
        # Reuse the transform from the last per-category dataset
        self.transform       = cat_ds.transform
        self.mask_transform  = cat_ds.mask_transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]
        image = self.transform(Image.open(img_path).convert('RGB'))
        H, W = image.shape[1], image.shape[2]
        mask = torch.zeros(1, H, W)   # train split → always normal, no mask
        return image, mask, label


# ---------------------------------------------------------------------------
# MVTec AD 2
# ---------------------------------------------------------------------------

class MVTec2Dataset(Dataset):
    """
    MVTec AD 2 dataset loader.

    Directory layout expected:
      <root>/<category>/train/             normal images (flat)
      <root>/<category>/test_public/good/  normal test images
      <root>/<category>/test_public/bad/   anomalous test images
      <root>/<category>/ground_truth/bad/  pixel masks (*_mask.png or *.png)

    split='train' → normal training images only.
    split='test'  → test_public/good (label 0) + test_public/bad (label 1).
    """

    _EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif')

    def __init__(self, root: str, category: str, split: str = 'train', img_size: int = 224):
        self.root = Path(root)
        self.transform = get_transform(img_size)
        self.mask_transform = T.Compose([
            T.Resize((img_size, img_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor(),
        ])
        self.samples: list = []
        self._load(split, category)

    def _load(self, split, category):
        cat_dir = self.root / category
        if split == 'train':
            for p in sorted((cat_dir / 'train').glob('*')):
                if p.suffix.lower() in self._EXTS:
                    self.samples.append((p, None, 0))
        else:
            good_dir = cat_dir / 'test_public' / 'good'
            if good_dir.exists():
                for p in sorted(good_dir.glob('*')):
                    if p.suffix.lower() in self._EXTS:
                        self.samples.append((p, None, 0))
            bad_dir = cat_dir / 'test_public' / 'bad'
            gt_dir  = cat_dir / 'ground_truth' / 'bad'
            if bad_dir.exists():
                for p in sorted(bad_dir.glob('*')):
                    if p.suffix.lower() not in self._EXTS:
                        continue
                    mask = None
                    for candidate_name in (p.stem + '_mask.png', p.stem + '.png', p.name):
                        cand = gt_dir / candidate_name
                        if cand.exists():
                            mask = cand
                            break
                    self.samples.append((p, mask, 1))

        if len(self.samples) == 0:
            msg = f"No samples found for category '{category}', split '{split}'.\n"
            if not self.root.exists():
                msg += f"Root directory '{self.root}' does not exist. Please check your data_path setting."
            elif not cat_dir.exists():
                subdirs = [p.name for p in self.root.iterdir() if p.is_dir() and not p.name.startswith('.')]
                msg += f"Category directory '{category}' not found under root '{self.root}'. Available directories: {subdirs}"
            else:
                target_dir = cat_dir / 'train' if split == 'train' else cat_dir / 'test_public'
                if not target_dir.exists():
                    msg += f"Target directory '{target_dir}' does not exist."
                else:
                    all_files = [p.name for p in target_dir.glob('*') if not p.name.startswith('.')][:5]
                    msg += f"Directory exists but no matching files with extensions {self._EXTS}. Found files: {all_files}"
            raise ValueError(msg)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]
        image = self.transform(Image.open(img_path).convert('RGB'))
        H, W = image.shape[1], image.shape[2]
        if mask_path is not None:
            mask = self.mask_transform(Image.open(mask_path).convert('L'))
            mask = (mask > 0.5).float()
        else:
            mask = torch.zeros(1, H, W)
        return image, mask, label


# ---------------------------------------------------------------------------
# VisA
# ---------------------------------------------------------------------------

class VisADataset(Dataset):
    """
    VisA (Visual Anomaly) dataset loader.

    Directory layout expected:
      <root>/<category>/Data/Images/Normal/   normal images
      <root>/<category>/Data/Images/Anomaly/  anomalous images
      <root>/<category>/Data/Masks/Anomaly/   pixel masks (same stem as anomaly img, .png)

    VisA has no official train/test split for normal images, so we apply a
    deterministic 80/20 split on the sorted list of normal images:
      split='train' → first 80% of normal images
      split='test'  → last 20% of normal images + all anomaly images

    All anomaly images (with their masks) always go to the test split.
    """

    TRAIN_RATIO = 0.8
    _EXTS = ('.png', '.jpg', '.jpeg', '.bmp')

    def __init__(self, root: str, category: str, split: str = 'train', img_size: int = 224):
        self.root = Path(root)
        self.transform = get_transform(img_size)
        self.mask_transform = T.Compose([
            T.Resize((img_size, img_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor(),
        ])
        self.samples: list = []
        self._load(split, category)

    def _load(self, split, category):
        data_dir   = self.root / category / 'Data'
        normal_dir = data_dir / 'Images' / 'Normal'
        anomaly_dir = data_dir / 'Images' / 'Anomaly'
        mask_dir    = data_dir / 'Masks'  / 'Anomaly'

        # Sorted for determinism; case-insensitive extension check
        all_normal = sorted(
            p for p in normal_dir.glob('*') if p.suffix.lower() in self._EXTS
        )
        n_train = int(len(all_normal) * self.TRAIN_RATIO)

        if split == 'train':
            for p in all_normal[:n_train]:
                self.samples.append((p, None, 0))
        else:
            # Last 20% normal (test normal)
            for p in all_normal[n_train:]:
                self.samples.append((p, None, 0))
            # All anomaly images
            for p in sorted(
                q for q in anomaly_dir.glob('*') if q.suffix.lower() in self._EXTS
            ):
                mask = mask_dir / (p.stem + '.png')
                self.samples.append((p, mask if mask.exists() else None, 1))

        if len(self.samples) == 0:
            msg = f"No samples found for category '{category}', split '{split}'.\n"
            if not self.root.exists():
                msg += f"Root directory '{self.root}' does not exist. Please check your data_path setting."
            elif not (self.root / category).exists():
                subdirs = [p.name for p in self.root.iterdir() if p.is_dir() and not p.name.startswith('.')]
                msg += f"Category directory '{category}' not found under root '{self.root}'. Available directories: {subdirs}"
            elif not data_dir.exists():
                msg += f"Data directory '{data_dir}' does not exist under '{self.root / category}'."
            else:
                msg += f"Checked normal_dir={normal_dir}. Directory exists={normal_dir.exists()}."
            raise ValueError(msg)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]
        image = self.transform(Image.open(img_path).convert('RGB'))
        H, W = image.shape[1], image.shape[2]
        if mask_path is not None:
            mask = self.mask_transform(Image.open(mask_path).convert('L'))
            mask = (mask > 0.5).float()
        else:
            mask = torch.zeros(1, H, W)
        return image, mask, label


# ---------------------------------------------------------------------------
# Unified factory
# ---------------------------------------------------------------------------

_REGISTRY = {
    'mvtec':  (MVTecDataset,   MVTEC_CATEGORIES),
    'mvtec2': (MVTec2Dataset,  MVTEC2_CATEGORIES),
    'visa':   (VisADataset,    VISA_CATEGORIES),
}

ALL_CATEGORIES = {name: cats for name, (_, cats) in _REGISTRY.items()}


def make_dataset(
    dataset_name: str,
    root: str,
    categories: Union[str, List[str]],
    split: str,
    img_size: int = 224,
) -> Dataset:
    """
    Factory that returns a Dataset (or ConcatDataset for multiple categories).

    Parameters
    ----------
    dataset_name : 'mvtec' | 'mvtec2' | 'visa'
    root         : path to the dataset root directory
    categories   : single category name, 'all', or list of category names
    split        : 'train' | 'test'
    img_size     : images are resized to (img_size x img_size)

    Returns
    -------
    A Dataset whose __getitem__ yields (image [3,H,W], mask [1,H,W], label).
    """
    if dataset_name not in _REGISTRY:
        raise ValueError(
            f"Unknown dataset '{dataset_name}'. Valid options: {list(_REGISTRY)}"
        )
    cls, all_cats = _REGISTRY[dataset_name]

    if isinstance(categories, str):
        categories = all_cats if categories == 'all' else [c.strip() for c in categories.split(',')]

    datasets = [cls(root, cat, split=split, img_size=img_size) for cat in categories]
    if len(datasets) == 1:
        return datasets[0]
    return ConcatDataset(datasets)


## 3. Training Logic

We define `train_category` (for single-category models) and `train_all` (for models trained jointly across multiple categories). Only the `predictor` sub-network parameters are optimized; the DINOv2 encoder is kept frozen.

In [ ]:
# Training logic adapted from train.py

def train_category(config, dataset_name: str, category: str):
    print(f'\n{"="*60}')
    print(f'  Training [{dataset_name}]: {category}')
    print(f'{"="*60}')

    device = torch.device(config.device)

    # Dataset
    dataset = make_dataset(dataset_name, config.train_data_path, category,
                           split='train', img_size=config.img_size)
    loader  = DataLoader(dataset, batch_size=config.batch_size, shuffle=True,
                         num_workers=4, pin_memory=True, drop_last=True)
    print(f'  Training images: {len(dataset)}')

    # Model — encoder is frozen inside PPAD
    model = PPAD(patch_grid=config.patch_grid, img_size=config.img_size,
                 encoder_name=config.encoder).to(device)

    # Only optimise predictor parameters
    optimizer = torch.optim.AdamW(model.predictor.parameters(), lr=config.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    best_loss = float('inf')
    out_dir   = Path(config.output_dir) / dataset_name / category
    out_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, config.epochs + 1):
        model.train()
        epoch_loss = 0.0

        for images, _, _ in tqdm(loader, desc=f'Epoch {epoch}/{config.epochs}', leave=False):
            images = images.to(device)
            B, N   = images.shape[0], model.num_patches

            # ---- encode (no grad, encoder is frozen) ----
            hp = model._encode_all_patches(images)    # [B, N, D]
            hi = model._encode_all_contexts(images)   # [B, N, D]

            # ---- predict (differentiable) ----
            hi_flat  = hi.reshape(B * N, -1)
            idx_flat = torch.arange(N, device=device).repeat(B)
            ho_flat  = model.predictor(hi_flat, idx_flat)            # [B*N, D]

            ho = ho_flat.view(B, N, -1)                              # [B, N, D]

            # Cosine-similarity loss on normalized vectors
            loss = (1.0 - F.cosine_similarity(ho, hp, dim=-1)).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.predictor.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        print(f'  Epoch {epoch:3d} | loss: {avg_loss:.4f} | lr: {scheduler.get_last_lr()[0]:.2e}')

        if avg_loss < best_loss:
            best_loss = avg_loss
            ckpt = {
                'epoch':       epoch,
                'loss':        best_loss,
                'predictor':   model.predictor.state_dict(),
                'patch_grid':  config.patch_grid,
                'img_size':    config.img_size,
                'encoder':     config.encoder,
                'dataset':     dataset_name,
                'categories':  [category],
            }
            torch.save(ckpt, out_dir / 'best.pt')

    print(f'  Saved best checkpoint → {out_dir / "best.pt"}  (loss={best_loss:.4f})')


def train_all(config, dataset_name: str, categories: list):
    all_cats = ALL_CATEGORIES[dataset_name]
    tag = 'all' if set(categories) == set(all_cats) else '+'.join(categories)
    print(f'\n{"="*60}')
    print(f'  Training ONE shared [{dataset_name}] model on: {categories}')
    print(f'{"="*60}')

    device = torch.device(config.device)

    # Combined dataset via factory
    dataset = make_dataset(dataset_name, config.train_data_path, categories,
                           split='train', img_size=config.img_size)
    loader  = DataLoader(dataset, batch_size=config.batch_size, shuffle=True,
                         num_workers=4, pin_memory=True, drop_last=True)
    print(f'  Total training images: {len(dataset)}')

    # One model for all categories
    model = PPAD(patch_grid=config.patch_grid, img_size=config.img_size,
                 encoder_name=config.encoder).to(device)

    optimizer = torch.optim.AdamW(model.predictor.parameters(), lr=config.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    best_loss = float('inf')
    out_dir   = Path(config.output_dir) / dataset_name / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, config.epochs + 1):
        model.train()
        epoch_loss = 0.0

        for images, _, _ in tqdm(loader, desc=f'Epoch {epoch}/{config.epochs}', leave=False):
            images = images.to(device)
            B, N   = images.shape[0], model.num_patches

            hp = model._encode_all_patches(images)    # [B, N, D]
            hi = model._encode_all_contexts(images)   # [B, N, D]

            hi_flat  = hi.reshape(B * N, -1)
            idx_flat = torch.arange(N, device=device).repeat(B)
            ho_flat  = model.predictor(hi_flat, idx_flat)

            ho   = ho_flat.view(B, N, -1)
            loss = (1.0 - F.cosine_similarity(ho, hp, dim=-1)).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.predictor.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        print(f'  Epoch {epoch:3d} | loss: {avg_loss:.4f} | lr: {scheduler.get_last_lr()[0]:.2e}')

        if avg_loss < best_loss:
            best_loss = avg_loss
            ckpt = {
                'epoch':      epoch,
                'loss':       best_loss,
                'predictor':  model.predictor.state_dict(),
                'patch_grid': config.patch_grid,
                'img_size':   config.img_size,
                'encoder':    config.encoder,
                'dataset':    dataset_name,
                'categories': categories,
            }
            torch.save(ckpt, out_dir / 'best.pt')

    print(f'  Saved best checkpoint → {out_dir / "best.pt"}  (loss={best_loss:.4f})')


def run_training(config):
    dataset_name = config.train_dataset

    # Resolve categories
    all_cats = ALL_CATEGORIES[dataset_name]
    if config.train_category == 'all':
        categories = all_cats
    else:
        categories = [c.strip() for c in config.train_category.split(',')]

    if len(categories) == 1:
        # Single category → per-category checkpoint
        train_category(config, dataset_name, categories[0])
    else:
        # Multiple categories → one shared model
        train_all(config, dataset_name, categories)

## 4. Evaluation and Visualizations

This section defines evaluation helpers:
- `save_visualization`: Generates a three-panel plot (Original Image | Ground Truth Mask | Predicted Anomaly Map) and optionally displays it directly in the notebook.
- `evaluate_category`: Evaluates the model on a test set, computing image-level and pixel-level AUROC, optimal pixel-level segmentation F1-score, and plotting qualitative heatmaps.

In [ ]:
# Evaluation logic adapted from evaluate.py

_IMG_MEAN = np.array([0.485, 0.456, 0.406])
_IMG_STD  = np.array([0.229, 0.224, 0.225])

def _denorm(tensor):
    """[3, H, W] normalized tensor → [H, W, 3] float32 in [0, 1]."""
    img = tensor.permute(1, 2, 0).cpu().float().numpy()
    return np.clip(img * _IMG_STD + _IMG_MEAN, 0.0, 1.0)


def save_visualization(img_tensor, mask_tensor, heatmap_tensor,
                       label: int, img_score: float, save_path: Path = None, show_inline: bool = False):
    """
    Save or display a 3-panel figure:
      Left   : original image (denormalized)
      Middle : ground-truth mask (gray)
      Right  : anomaly heatmap (hot colormap) blended over the image
    """
    img_np  = _denorm(img_tensor)                          # [H, W, 3]
    mask_np = mask_tensor.cpu().numpy()                    # [H, W]
    heat_np = heatmap_tensor.cpu().numpy()                 # [H, W]

    # Normalize heatmap to [0, 1] for display
    h_min, h_max = heat_np.min(), heat_np.max()
    heat_norm = (heat_np - h_min) / (h_max - h_min + 1e-8)

    # Blend heatmap over image (alpha composite)
    cmap   = cm.get_cmap('hot')
    heat_rgb = cmap(heat_norm)[..., :3]                    # [H, W, 3]
    blend    = 0.55 * img_np + 0.45 * heat_rgb             # [H, W, 3]
    blend    = np.clip(blend, 0.0, 1.0)

    status = 'ANOMALY' if label == 1 else 'NORMAL'
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5),
                             gridspec_kw={'wspace': 0.05})
    fig.patch.set_facecolor('#1a1a2e')

    # Panel 1 — image
    axes[0].imshow(img_np)
    axes[0].set_title('Image', color='white', fontsize=11, pad=6)
    axes[0].axis('off')

    # Panel 2 — GT mask
    axes[1].imshow(mask_np, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Ground Truth Mask', color='white', fontsize=11, pad=6)
    axes[1].axis('off')

    # Panel 3 — anomaly map blended
    axes[2].imshow(blend)
    axes[2].set_title(
        f'Anomaly Map  [{status}]  score={img_score:.3f}',
        color='tomato' if label == 1 else 'lightgreen',
        fontsize=11, pad=6,
    )
    axes[2].axis('off')

    # Colorbar for anomaly map
    sm = plt.cm.ScalarMappable(cmap='hot', norm=Normalize(vmin=h_min, vmax=h_max))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[2], fraction=0.046, pad=0.04)
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=8)

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=130, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
    
    if show_inline:
        plt.show()
    else:
        plt.close(fig)


def evaluate_category(config, dataset_name: str, data_path: str, category: str, ckpt_tag: str) -> dict:
    device = torch.device(config.device)

    # Resolve checkpoint path
    ckpt_path = Path(config.output_dir) / ckpt_tag / 'best.pt'
    if not ckpt_path.exists():
        print(f'  [SKIP] No checkpoint found: {ckpt_path}')
        return {}

    ckpt = torch.load(ckpt_path, map_location=device)

    model = PPAD(
        patch_grid   = ckpt['patch_grid'],
        img_size     = ckpt['img_size'],
        encoder_name = ckpt['encoder'],
    ).to(device)
    model.predictor.load_state_dict(ckpt['predictor'])
    model.eval()

    # Dataset — use the appropriate loader for the *test* dataset
    dataset = make_dataset(dataset_name, data_path, category,
                           split='test', img_size=ckpt['img_size'])
    loader  = DataLoader(dataset, batch_size=1, shuffle=False,
                         num_workers=4, pin_memory=True)

    # Visualizations go under <vis_dir>/<dataset>/<category>/
    vis_dir = Path(config.vis_dir) / dataset_name / category
    if not config.no_vis:
        vis_dir.mkdir(parents=True, exist_ok=True)
        print(f'  Saving visualizations → {vis_dir}')

    img_scores, img_labels = [], []
    px_scores,  px_labels  = [], []

    with torch.no_grad():
        for sample_idx, (images, masks, labels) in enumerate(
                tqdm(loader, desc=f'  [{dataset_name}] {category}')):
            images = images.to(device)          # [1, 3, H, W]

            # Anomaly map [1, 1, H, W]
            heatmap = model.get_anomaly_map(images)

            # Image-level score = max over spatial positions
            img_score = heatmap.flatten(1).max(dim=1).values  # [1]

            img_scores.append(img_score.cpu().numpy())
            img_labels.append(labels.numpy())

            # Pixel-level
            px_scores.append(heatmap.squeeze(1).cpu().numpy())   # [1, H, W]
            px_labels.append(masks.squeeze(1).cpu().numpy())     # [1, H, W]

            # ---- Per-image visualization --------------------------------
            # We display the first 3 samples inline regardless of no_vis
            show_inline = (sample_idx < 3)
            save_path = None
            if not config.no_vis:
                label_int  = int(labels[0])
                score_val  = float(img_score[0])
                status_str = 'anomaly' if label_int == 1 else 'normal'
                fname      = f'{sample_idx:04d}_{status_str}.png'
                save_path  = vis_dir / fname

            if show_inline or save_path is not None:
                label_int  = int(labels[0])
                score_val  = float(img_score[0])
                save_visualization(
                    img_tensor     = images[0],
                    mask_tensor    = masks[0, 0],
                    heatmap_tensor = heatmap[0, 0],
                    label          = label_int,
                    img_score      = score_val,
                    save_path      = save_path,
                    show_inline    = show_inline,
                )

    img_scores = np.concatenate(img_scores)
    img_labels = np.concatenate(img_labels)
    px_scores  = np.concatenate(px_scores).ravel()
    px_labels  = np.concatenate(px_labels).ravel().astype(int)

    img_auroc = roc_auc_score(img_labels, img_scores) * 100
    px_auroc  = roc_auc_score(px_labels,  px_scores)  * 100 if px_labels.sum() > 0 else float('nan')

    # Seg-F1: find optimal threshold per category from GT pixel labels
    if px_labels.sum() > 0:
        precision, recall, thresholds = precision_recall_curve(px_labels, px_scores)
        f1_per_thresh = 2 * precision * recall / (precision + recall + 1e-8)
        best_idx  = int(np.argmax(f1_per_thresh))
        seg_f1    = float(f1_per_thresh[best_idx]) * 100
        threshold = float(thresholds[min(best_idx, len(thresholds) - 1)])
    else:
        seg_f1    = float('nan')
        threshold = float('nan')

    print(f'  {category:<15} img-AUROC: {img_auroc:.1f}%   px-AUROC: {px_auroc:.1f}%   seg-F1: {seg_f1:.1f}%  (thr={threshold:.4f})')
    return {'img_auroc': img_auroc, 'px_auroc': px_auroc, 'seg_f1': seg_f1, 'threshold': threshold}


def run_evaluation(config):
    dataset_name = config.eval_dataset
    data_path    = config.eval_data_path

    # Resolve categories
    all_cats = ALL_CATEGORIES[dataset_name]
    if config.eval_category == 'all':
        categories = all_cats
    else:
        categories = [c.strip() for c in config.eval_category.split(',')]

    # Resolve checkpoint tag to load
    if config.ckpt_tag:
        tag = config.ckpt_tag
    elif config.train:
        # Default tag from the training session in this run
        if config.train_category == 'all':
            tag = f'{config.train_dataset}/all'
        else:
            t_cats = [c.strip() for c in config.train_category.split(',')]
            tag = f'{config.train_dataset}/{t_cats[0]}' if len(t_cats) == 1                   else f'{config.train_dataset}/all'
    else:
        # Assume standard local path for eval_dataset
        tag = f'{dataset_name}/{categories[0]}' if len(categories) == 1               else f'{dataset_name}/all'

    print(f'\n{"="*60}')
    print(f'  PPAD Evaluation  |  dataset: {dataset_name}')
    print(f'  ckpt_dir: {config.output_dir}')
    print(f'  ckpt_tag: {tag}')
    print(f'{"="*60}')

    results = {}
    for cat in categories:
        results[cat] = evaluate_category(config, dataset_name, data_path, cat, tag)

    # Summary
    valid = {k: v for k, v in results.items() if v}
    if len(valid) > 1:
        mean_img = np.mean([v['img_auroc'] for v in valid.values()])
        mean_px  = np.mean([v['px_auroc'] for v in valid.values()
                            if not np.isnan(v['px_auroc'])])
        mean_f1  = np.mean([v['seg_f1']   for v in valid.values()
                            if not np.isnan(v['seg_f1'])])
        print(f'\n  Mean img-AUROC: {mean_img:.1f}%   Mean px-AUROC: {mean_px:.1f}%   Mean seg-F1: {mean_f1:.1f}%')

    # Save results JSON next to the checkpoint dir
    out = Path(config.output_dir) / 'results.json'
    with open(out, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\n  Results saved → {out}')
    return results

## 5. Execute Training & Evaluation

We create any necessary directories, then call training and evaluation loops based on your setup.

In [ ]:
# Create directories
os.makedirs(Config.output_dir, exist_ok=True)
os.makedirs(Config.vis_dir, exist_ok=True)

# Run Training
if Config.train:
    run_training(Config)

# Run Evaluation
if Config.evaluate:
    run_evaluation(Config)